# Tentativi random di variational bayes

In [1]:
import torch
import torch.nn as nn

In [2]:
class Reservoir(nn.Module):
    def __init__(self, N, K, spectral_radius=0.9):
        super(Reservoir, self).__init__()
        
        self.N = N  # Number of reservoir neurons
        self.K = K  # Input dimension

        
        # Initialize Input Matrix (Uniform distribution is common)
        W_in = torch.rand(N, K) * 2 - 1  # Range [-1, 1]
        
        # Initialize Reservoir Matrix (Gaussian)
        W = torch.randn(N, N)
        
        # Scale Spectral Radius for stability
        # We calculate the largest eigenvalue and scale W
        with torch.no_grad():
            # Use real/imaginary parts to find the magnitude of eigenvalues
            eigenvalues = torch.linalg.eigvals(W)
            max_eig = torch.max(torch.abs(eigenvalues))
            W = W * (spectral_radius / max_eig)
        
        # Register as buffers (Fixed weights, move with model to GPU)
        self.register_buffer('W_in', W_in)
        self.register_buffer('W', W)
        
        # Internal state (Initialized to zeros)
        self.register_buffer('states', torch.zeros(1, N))
        
        self.activation = torch.tanh

    def forward(self, x):
        """
        x shape: [Batch, K]
        Returns the updated state: [Batch, N]
        """
        
        # Linear combinations: Input effect + Reservoir recurrent effect
        # We use .t() on weights because x is [Batch, K] and states is [Batch, N]
        input_part = x @ self.W_in.t()
        recurrent_part = self.states @ self.W.t()
        
        # We .detach() to ensure we don't track gradients through time steps
        self.states = self.activation(input_part + recurrent_part).detach()
        
        return self.states

    def reset_state(self, batch_size=1):
        """Clears the memory of the reservoir."""
        self.states = torch.zeros(batch_size, self.N, device=self.W.device)

The output matrix of the states update has dimension (batch_size,N)

In [5]:
L = 100  
K = 1    # Input dimension (1D series)
N = 50   # Number of neurons in the reservoir (state dimension)

# Create an example time series (a sine wave with noise)
t = torch.linspace(0, 10, L)
time_series = torch.sin(t) + torch.randn(L) * 0.1

# Reshape to [L, 1, K] to represent [Length, Batch=1, Input_dim]
# This format is standard for processing sequences in PyTorch
time_series = time_series.view(L, 1, 1)

# Initialize the model
model = Reservoir(N=N, K=K)
model.reset_state(batch_size=1)

# Container to collect states
# We want to save a state of dimension N for each time step L
collected_states = torch.zeros(L, N)

# Temporal Loop
# In an ESN, we process the series one step at a time to update the memory
for t in range(L):
    x_t = time_series[t]  # Get the input at time t: [1, 1]
    
    # The forward pass updates model.states and returns the current state
    current_state = model(x_t) # output shape: [1, N]
    
    # Save the state into our container
    # .view(-1) flattens the [1, N] tensor into [N] to fit the row
    collected_states[t] = current_state.view(-1)

# Now 'collected_states' is a [100 x 50] matrix
print(f"State matrix shape: {collected_states.shape}")

State matrix shape: torch.Size([100, 50])
